## Product Intelligence

### Objetivo

El objetivo de este notebook es analizar el rendimiento de los productos desde una perspectiva de merchandising, incorporando el efecto de las devoluciones.

La pregunta principal es:

**¿Los productos que parecen tener mejor rendimiento cuando se analiza la demanda bruta mantienen esa posición cuando se observa la demanda realmente retenida?**

El análisis se centrará en:

- comparar unidades pedidas frente a unidades retenidas;
- identificar productos con altas tasas de devolución;
- detectar cambios relevantes en el ranking de producto;
- analizar el rendimiento por variantes de talla y color;
- estudiar la relación entre pricing, markdown y devoluciones;
- identificar productos que requieren atención de Merchandising.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROCESSED_PATH = Path("../data/processed")

df = pd.read_parquet(
    PROCESSED_PATH / "analytical_transactions.parquet"
)

df.shape

(2325165, 32)

In [3]:
df["is_valid_transaction"].value_counts()

is_valid_transaction
True     2293521
False      31644
Name: count, dtype: int64

In [4]:
df[
    [
        "ordered_units",
        "returned_units",
        "kept_units",
        "ordered_sales_value",
        "returned_sales_value",
        "kept_sales_value"
    ]
].head()

,ordered_units,returned_units,kept_units,ordered_sales_value,returned_sales_value,kept_sales_value
0,1,0,1,10.00,0.00,10.0
1,1,0,1,20.00,0.00,20.0
2,1,0,1,35.00,0.00,35.0
3,1,1,0,49.99,49.99,0.0
4,1,0,1,10.00,0.00,10.0


In [5]:
product_base = df.loc[
    df["is_valid_transaction"]
    & ~df["is_zero_value_item"]
].copy()

In [6]:
product_base.shape

(2278321, 32)

In [7]:
product_performance = (
    product_base
    .groupby("articleID")
    .agg(
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        kept_units=("kept_units", "sum"),
        
        ordered_sales_value=("ordered_sales_value", "sum"),
        returned_sales_value=("returned_sales_value", "sum"),
        kept_sales_value=("kept_sales_value", "sum"),
        
        transactions=("orderID", "size"),
        customers=("customerID", "nunique")
    )
    .reset_index()
)

In [8]:
product_performance.shape

(3807, 9)

In [9]:
product_performance.head()

,articleID,ordered_units,returned_units,kept_units,ordered_sales_value,returned_sales_value,kept_sales_value,transactions,customers
0,i1000003,1,1,0,20.0,20.0,0.0,1,1
1,i1000005,1,1,0,10.0,10.0,0.0,1,1
2,i1000006,1,1,0,20.0,20.0,0.0,1,1
3,i1000007,2,1,1,40.0,20.0,20.0,2,2
4,i1000008,6,5,1,90.0,75.0,15.0,6,6


In [10]:
product_performance["unit_return_rate"] = (
    product_performance["returned_units"]
    / product_performance["ordered_units"]
)

product_performance["unit_retention_rate"] = (
    product_performance["kept_units"]
    / product_performance["ordered_units"]
)

In [11]:
product_performance["value_return_rate"] = (
    product_performance["returned_sales_value"]
    / product_performance["ordered_sales_value"]
)

product_performance["value_retention_rate"] = (
    product_performance["kept_sales_value"]
    / product_performance["ordered_sales_value"]
)

In [12]:
product_performance[
    [
        "ordered_units",
        "returned_units",
        "kept_units",
        "unit_return_rate",
        "unit_retention_rate"
    ]
].describe()

,ordered_units,returned_units,kept_units,unit_return_rate,unit_retention_rate
count,3807.000000,3807.000000,3807.000000,3807.000000,3807.000000
mean,604.044392,318.031784,286.012608,0.541316,0.458684
std,948.223685,417.838482,582.809806,0.145199,0.145199
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,144.000000,75.000000,63.000000,0.464395,0.368576
50%,364.000000,199.000000,151.000000,0.552124,0.447876
75%,783.500000,426.000000,345.000000,0.631424,0.535605
max,27611.000000,10026.000000,20552.000000,1.000000,1.000000


In [13]:
product_performance["articleID"].nunique()

3807

In [14]:
min_ordered_units = product_performance["ordered_units"].quantile(0.25)

min_ordered_units

np.float64(144.0)

In [15]:
product_analysis = product_performance.loc[
    product_performance["ordered_units"] >= min_ordered_units
].copy()

product_analysis.shape

(2862, 13)

In [16]:
coverage_units = (
    product_analysis["ordered_units"].sum()
    / product_performance["ordered_units"].sum()
)

coverage_value = (
    product_analysis["ordered_sales_value"].sum()
    / product_performance["ordered_sales_value"].sum()
)

coverage_units, coverage_value

(np.float64(0.9770929428069353), np.float64(0.9753037903681002))

In [17]:
product_analysis["ordered_units_rank"] = (
    product_analysis["ordered_units"]
    .rank(method="min", ascending=False)
)

product_analysis["kept_units_rank"] = (
    product_analysis["kept_units"]
    .rank(method="min", ascending=False)
)

In [18]:
product_analysis["rank_drop"] = (
    product_analysis["kept_units_rank"]
    - product_analysis["ordered_units_rank"]
)

In [19]:
largest_rank_drops = (
    product_analysis
    .sort_values("rank_drop", ascending=False)
    [
        [
            "articleID",
            "ordered_units",
            "returned_units",
            "kept_units",
            "unit_return_rate",
            "ordered_units_rank",
            "kept_units_rank",
            "rank_drop"
        ]
    ]
    .head(20)
)

largest_rank_drops

,articleID,ordered_units,returned_units,kept_units,unit_return_rate,ordered_units_rank,kept_units_rank,rank_drop
1955,i1002046,1104,969,135,0.877717,560.0,2050.0,1490.0
3368,i1003520,831,710,121,0.854392,886.0,2186.0,1300.0
3704,i1003872,517,443,74,0.856867,1442.0,2664.0,1222.0
2155,i1002257,523,439,84,0.839388,1427.0,2571.0,1144.0
2204,i1002311,620,514,106,0.829032,1227.0,2355.0,1128.0
3231,i1003383,481,403,78,0.837838,1534.0,2620.0,1086.0
154,i1000168,600,488,112,0.813333,1262.0,2283.0,1021.0
1723,i1001812,800,650,150,0.812500,935.0,1915.0,980.0
2167,i1002269,583,468,115,0.802744,1293.0,2252.0,959.0
153,i1000167,654,526,128,0.804281,1159.0,2109.0,950.0


In [20]:
largest_rank_improvements = (
    product_analysis
    .sort_values("rank_drop")
    [
        [
            "articleID",
            "ordered_units",
            "returned_units",
            "kept_units",
            "unit_return_rate",
            "ordered_units_rank",
            "kept_units_rank",
            "rank_drop"
        ]
    ]
    .head(20)
)

largest_rank_improvements

,articleID,ordered_units,returned_units,kept_units,unit_return_rate,ordered_units_rank,kept_units_rank,rank_drop
3806,i1004001,351,0,351,0.000000,1943.0,931.0,-1012.0
2939,i1003074,312,69,243,0.221154,2084.0,1342.0,-742.0
2807,i1002938,297,67,230,0.225589,2153.0,1413.0,-740.0
2828,i1002959,332,79,253,0.237952,2000.0,1285.0,-715.0
2601,i1002732,305,74,231,0.242623,2114.0,1401.0,-713.0
2543,i1002674,313,77,236,0.246006,2080.0,1378.0,-702.0
2626,i1002757,352,88,264,0.250000,1937.0,1240.0,-697.0
2658,i1002789,411,99,312,0.240876,1743.0,1057.0,-686.0
2725,i1002856,209,54,155,0.258373,2539.0,1860.0,-679.0
2897,i1003031,249,68,181,0.273092,2377.0,1698.0,-679.0


In [21]:
product_analysis["abs_rank_shift"] = (
    product_analysis["rank_drop"].abs()
)

product_analysis["abs_rank_shift"].describe()

count    2862.000000
mean      201.395877
std       181.634426
min         0.000000
25%        61.250000
50%       150.000000
75%       292.750000
max      1490.000000
Name: abs_rank_shift, dtype: float64

In [22]:
rank_correlation = product_analysis[
    ["ordered_units_rank", "kept_units_rank"]
].corr(method="spearman").iloc[0, 1]

rank_correlation

np.float64(0.9460228759614362)

In [23]:
TOP_N = 100

top_ordered = set(
    product_analysis
    .nlargest(TOP_N, "ordered_units")["articleID"]
)

top_kept = set(
    product_analysis
    .nlargest(TOP_N, "kept_units")["articleID"]
)

top_100_overlap = len(
    top_ordered.intersection(top_kept)
)

top_100_overlap

81

In [24]:
top_100_overlap_pct = top_100_overlap / TOP_N

top_100_overlap_pct

0.81

In [25]:
products_leaving_top100 = (
    product_analysis[
        product_analysis["articleID"].isin(
            top_ordered - top_kept
        )
    ]
    .sort_values("ordered_units_rank")
    [
        [
            "articleID",
            "ordered_units",
            "returned_units",
            "kept_units",
            "unit_return_rate",
            "ordered_units_rank",
            "kept_units_rank",
            "rank_drop"
        ]
    ]
)

products_leaving_top100

,articleID,ordered_units,returned_units,kept_units,unit_return_rate,ordered_units_rank,kept_units_rank,rank_drop
2227,i1002334,3105,2502,603,0.805797,55.0,424.0,369.0
1788,i1001877,3042,2041,1001,0.670940,60.0,145.0,85.0
1592,i1001674,2928,1778,1150,0.607240,67.0,103.0,36.0
1724,i1001813,2821,1887,934,0.668912,71.0,175.0,104.0
3722,i1003890,2805,1709,1096,0.609269,74.0,117.0,43.0
1579,i1001661,2763,1758,1005,0.636265,77.0,142.0,65.0
1041,i1001082,2750,1683,1067,0.612000,79.0,128.0,49.0
1037,i1001078,2696,1616,1080,0.599407,80.0,124.0,44.0
3367,i1003519,2685,1549,1136,0.576909,81.0,107.0,26.0
3718,i1003886,2684,1569,1115,0.584575,82.0,112.0,30.0


In [26]:
top100_analysis = product_analysis.loc[
    product_analysis["articleID"].isin(top_ordered)
].copy()

top100_analysis["stays_in_top100"] = (
    top100_analysis["articleID"].isin(top_kept)
)

In [27]:
top100_analysis.groupby(
    "stays_in_top100"
).agg(
    products=("articleID", "count"),
    avg_ordered_units=("ordered_units", "mean"),
    avg_returned_units=("returned_units", "mean"),
    avg_kept_units=("kept_units", "mean"),
    avg_return_rate=("unit_return_rate", "mean"),
    median_return_rate=("unit_return_rate", "median")
)

,products,avg_ordered_units,avg_returned_units,avg_kept_units,avg_return_rate,median_return_rate
stays_in_top100,,,,,,
False,19,2687.684211,1672.473684,1015.210526,0.620112,0.609269
True,81,4667.407407,1951.345679,2716.061728,0.443777,0.447856


In [28]:
leaving_top100 = top100_analysis.loc[
    ~top100_analysis["stays_in_top100"]
]

leaving_top100_summary = pd.Series({
    "products": leaving_top100["articleID"].nunique(),
    "ordered_units": leaving_top100["ordered_units"].sum(),
    "returned_units": leaving_top100["returned_units"].sum(),
    "kept_units": leaving_top100["kept_units"].sum(),
    "return_rate": (
        leaving_top100["returned_units"].sum()
        / leaving_top100["ordered_units"].sum()
    )
})

leaving_top100_summary

products             19.000000
ordered_units     51066.000000
returned_units    31777.000000
kept_units        19289.000000
return_rate           0.622273
dtype: float64

In [29]:
top100_group_summary = (
    top100_analysis
    .groupby("stays_in_top100")
    .agg(
        products=("articleID", "count"),
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        kept_units=("kept_units", "sum"),
        ordered_sales_value=("ordered_sales_value", "sum"),
        returned_sales_value=("returned_sales_value", "sum"),
        kept_sales_value=("kept_sales_value", "sum")
    )
)

top100_group_summary["weighted_unit_return_rate"] = (
    top100_group_summary["returned_units"]
    / top100_group_summary["ordered_units"]
)

top100_group_summary["weighted_value_return_rate"] = (
    top100_group_summary["returned_sales_value"]
    / top100_group_summary["ordered_sales_value"]
)

top100_group_summary

,products,ordered_units,returned_units,kept_units,ordered_sales_value,returned_sales_value,kept_sales_value,weighted_unit_return_rate,weighted_value_return_rate
stays_in_top100,,,,,,,,,
False,19,51066,31777,19289,1926183.87,1219263.89,706919.98,0.622273,0.632995
True,81,378060,158059,220001,9563843.69,4461691.50,5102152.19,0.418079,0.466517


In [30]:
product_group_check = (
    product_base
    .groupby("articleID")["productGroup"]
    .nunique(dropna=True)
)

product_group_check.value_counts()

productGroup
1    3806
0       1
Name: count, dtype: int64

In [31]:
article_group = (
    product_base[
        ["articleID", "productGroup"]
    ]
    .dropna()
    .drop_duplicates()
)

In [32]:
article_group.shape

(3806, 2)

In [33]:
top100_analysis = top100_analysis.merge(
    article_group,
    on="articleID",
    how="left"
)

In [34]:
top100_analysis.shape

(100, 19)

In [35]:
product_group_summary = (
    top100_analysis
    .groupby(
        ["stays_in_top100", "productGroup"],
        dropna=False
    )
    .agg(
        products=("articleID", "nunique"),
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        kept_units=("kept_units", "sum")
    )
    .reset_index()
)

In [36]:
product_group_summary["return_rate"] = (
    product_group_summary["returned_units"]
    / product_group_summary["ordered_units"]
)

In [37]:
product_group_summary.loc[
    product_group_summary["stays_in_top100"] == False
].sort_values(
    "products",
    ascending=False
)

,stays_in_top100,productGroup,products,ordered_units,returned_units,kept_units,return_rate
4,False,8.0,7,18849,11891,6958,0.630856
2,False,3.0,5,13010,7710,5300,0.592621
1,False,2.0,3,7850,4638,3212,0.590828
0,False,1.0,2,5489,3278,2211,0.597194
3,False,7.0,1,2763,1758,1005,0.636265
5,False,13.0,1,3105,2502,603,0.805797


In [38]:
leaving_group_counts = (
    top100_analysis.loc[
        ~top100_analysis["stays_in_top100"],
        "productGroup"
    ]
    .value_counts(dropna=False)
)

leaving_group_counts

productGroup
8.0     7
3.0     5
2.0     3
1.0     2
7.0     1
13.0    1
Name: count, dtype: int64

In [39]:
group_exit_analysis = (
    top100_analysis
    .groupby("productGroup", dropna=False)
    .agg(
        top100_products=("articleID", "nunique"),
        leaving_products=(
            "stays_in_top100",
            lambda x: (~x).sum()
        ),
        staying_products=(
            "stays_in_top100",
            "sum"
        )
    )
    .reset_index()
)

group_exit_analysis["exit_rate"] = (
    group_exit_analysis["leaving_products"]
    / group_exit_analysis["top100_products"]
)

group_exit_analysis.sort_values(
    "exit_rate",
    ascending=False
)

,productGroup,top100_products,leaving_products,staying_products,exit_rate
5,7.0,1,1,0,1.000000
8,13.0,1,1,0,1.000000
1,2.0,5,3,2,0.600000
6,8.0,19,7,12,0.368421
0,1.0,6,2,4,0.333333
2,3.0,54,5,49,0.092593
4,5.0,3,0,3,0.000000
3,4.0,7,0,7,0.000000
7,9.0,2,0,2,0.000000
9,14.0,1,0,1,0.000000


In [41]:
group_exit_analysis["share_of_top100"] = (
    group_exit_analysis["top100_products"]
    / group_exit_analysis["top100_products"].sum()
)

group_exit_analysis["share_of_leavers"] = (
    group_exit_analysis["leaving_products"]
    / group_exit_analysis["leaving_products"].sum()
)

group_exit_analysis["overrepresentation"] = (
    group_exit_analysis["share_of_leavers"]
    / group_exit_analysis["share_of_top100"]
)

group_exit_analysis.sort_values(
    "overrepresentation",
    ascending=False
)

,productGroup,top100_products,leaving_products,staying_products,exit_rate,share_of_top100,share_of_leavers,overrepresentation
5,7.0,1,1,0,1.000000,0.01,0.052632,5.263158
8,13.0,1,1,0,1.000000,0.01,0.052632,5.263158
1,2.0,5,3,2,0.600000,0.05,0.157895,3.157895
6,8.0,19,7,12,0.368421,0.19,0.368421,1.939058
0,1.0,6,2,4,0.333333,0.06,0.105263,1.754386
2,3.0,54,5,49,0.092593,0.54,0.263158,0.487329
4,5.0,3,0,3,0.000000,0.03,0.000000,0.000000
3,4.0,7,0,7,0.000000,0.07,0.000000,0.000000
7,9.0,2,0,2,0.000000,0.02,0.000000,0.000000
9,14.0,1,0,1,0.000000,0.01,0.000000,0.000000


In [42]:
group_performance = (
    product_base
    .groupby("productGroup", dropna=False)
    .agg(
        products=("articleID", "nunique"),
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        kept_units=("kept_units", "sum"),
        ordered_sales_value=("ordered_sales_value", "sum"),
        returned_sales_value=("returned_sales_value", "sum"),
        kept_sales_value=("kept_sales_value", "sum")
    )
    .reset_index()
)

group_performance["unit_return_rate"] = (
    group_performance["returned_units"]
    / group_performance["ordered_units"]
)

group_performance["value_return_rate"] = (
    group_performance["returned_sales_value"]
    / group_performance["ordered_sales_value"]
)

group_performance.sort_values(
    "unit_return_rate",
    ascending=False
)

,productGroup,products,ordered_units,returned_units,kept_units,ordered_sales_value,returned_sales_value,kept_sales_value,unit_return_rate,value_return_rate
9,13.0,178,99459,68634,30825,4958095.76,3482447.84,1475647.92,0.690073,0.702376
6,7.0,100,40734,25924,14810,4032440.88,2623857.22,1408583.66,0.636422,0.650687
14,50.0,35,8066,5077,2989,1443799.58,929314.79,514484.79,0.629432,0.643659
10,14.0,105,44486,27948,16538,3029598.04,1930534.23,1099063.81,0.628243,0.637225
8,9.0,152,73060,44191,28869,3080498.52,1881997.20,1198501.32,0.604859,0.610939
0,1.0,405,203166,122460,80706,9745528.69,5946651.34,3798877.35,0.602758,0.610193
7,8.0,378,324129,191380,132749,10739502.32,6416981.13,4322521.19,0.590444,0.597512
1,2.0,506,203883,119212,84671,11258172.34,6649356.17,4608816.17,0.584708,0.590625
4,5.0,140,82022,45078,36944,3334887.12,1876380.94,1458506.18,0.549584,0.562652
11,15.0,23,10847,5860,4987,607544.17,327402.55,280141.62,0.540242,0.538895


In [43]:
group_diagnostics = group_exit_analysis.merge(
    group_performance[
        [
            "productGroup",
            "products",
            "ordered_units",
            "unit_return_rate",
            "value_return_rate"
        ]
    ],
    on="productGroup",
    how="left",
    suffixes=("_top100", "_overall")
)

group_diagnostics.sort_values(
    "exit_rate",
    ascending=False
)

,productGroup,top100_products,leaving_products,staying_products,exit_rate,share_of_top100,share_of_leavers,overrepresentation,products,ordered_units,unit_return_rate,value_return_rate
5,7.0,1,1,0,1.000000,0.01,0.052632,5.263158,100,40734,0.636422,0.650687
8,13.0,1,1,0,1.000000,0.01,0.052632,5.263158,178,99459,0.690073,0.702376
1,2.0,5,3,2,0.600000,0.05,0.157895,3.157895,506,203883,0.584708,0.590625
6,8.0,19,7,12,0.368421,0.19,0.368421,1.939058,378,324129,0.590444,0.597512
0,1.0,6,2,4,0.333333,0.06,0.105263,1.754386,405,203166,0.602758,0.610193
2,3.0,54,5,49,0.092593,0.54,0.263158,0.487329,964,900257,0.462382,0.488177
4,5.0,3,0,3,0.000000,0.03,0.000000,0.000000,140,82022,0.549584,0.562652
3,4.0,7,0,7,0.000000,0.07,0.000000,0.000000,166,115037,0.520702,0.545109
7,9.0,2,0,2,0.000000,0.02,0.000000,0.000000,152,73060,0.604859,0.610939
9,14.0,1,0,1,0.000000,0.01,0.000000,0.000000,105,44486,0.628243,0.637225


In [44]:
leaving_product_ids = set(
    top100_analysis.loc[
        ~top100_analysis["stays_in_top100"],
        "articleID"
    ]
)

leaving_variants = product_base.loc[
    product_base["articleID"].isin(leaving_product_ids)
].copy()

In [45]:
leaving_variants["articleID"].nunique()

19

In [46]:
size_performance = (
    leaving_variants
    .groupby(
        ["articleID", "sizeCode"]
    )
    .agg(
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        kept_units=("kept_units", "sum")
    )
    .reset_index()
)

size_performance["return_rate"] = (
    size_performance["returned_units"]
    / size_performance["ordered_units"]
)

In [47]:
size_performance["ordered_units"].describe()

count    129.000000
mean     395.860465
std      174.544261
min       29.000000
25%      258.000000
50%      419.000000
75%      536.000000
max      732.000000
Name: ordered_units, dtype: float64

In [48]:
size_performance.shape

(129, 6)

In [49]:
color_performance = (
    leaving_variants
    .groupby(
        ["articleID", "colorCode"]
    )
    .agg(
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        kept_units=("kept_units", "sum")
    )
    .reset_index()
)

color_performance["return_rate"] = (
    color_performance["returned_units"]
    / color_performance["ordered_units"]
)

In [50]:
color_performance["ordered_units"].describe()

count      85.000000
mean      600.776471
std       296.758535
min       217.000000
25%       454.000000
50%       560.000000
75%       672.000000
max      2600.000000
Name: ordered_units, dtype: float64

In [51]:
color_performance.shape

(85, 6)

In [52]:
article_return_rate = (
    leaving_variants
    .groupby("articleID")
    .agg(
        article_ordered_units=("ordered_units", "sum"),
        article_returned_units=("returned_units", "sum")
    )
    .reset_index()
)

article_return_rate["article_return_rate"] = (
    article_return_rate["article_returned_units"]
    / article_return_rate["article_ordered_units"]
)

In [53]:
size_diagnostics = size_performance.merge(
    article_return_rate[
        ["articleID", "article_return_rate"]
    ],
    on="articleID",
    how="left"
)

In [54]:
size_diagnostics["return_rate_gap"] = (
    size_diagnostics["return_rate"]
    - size_diagnostics["article_return_rate"]
)

In [55]:
size_diagnostics_robust = size_diagnostics.loc[
    size_diagnostics["ordered_units"] >= 50
].copy()

In [56]:
size_diagnostics_robust.sort_values(
    "return_rate_gap",
    ascending=False
)[
    [
        "articleID",
        "sizeCode",
        "ordered_units",
        "returned_units",
        "return_rate",
        "article_return_rate",
        "return_rate_gap"
    ]
].head(20)

,articleID,sizeCode,ordered_units,returned_units,return_rate,article_return_rate,return_rate_gap
122,i1003890,32,81,59,0.728395,0.609269,0.119126
28,i1001078,34,190,132,0.694737,0.599407,0.095330
121,i1003886,44,364,244,0.670330,0.584575,0.085754
22,i1000868,34,165,108,0.654545,0.573958,0.080588
52,i1001774,34,104,84,0.807692,0.729652,0.078040
123,i1003890,34,239,164,0.686192,0.609269,0.076923
64,i1001813,34,171,127,0.742690,0.668912,0.073778
10,i1000783,34,172,110,0.639535,0.566067,0.073468
107,i1003519,27,258,162,0.627907,0.576909,0.050998
108,i1003519,28,328,205,0.625000,0.576909,0.048091


In [57]:
color_diagnostics = color_performance.merge(
    article_return_rate[
        ["articleID", "article_return_rate"]
    ],
    on="articleID",
    how="left"
)

color_diagnostics["return_rate_gap"] = (
    color_diagnostics["return_rate"]
    - color_diagnostics["article_return_rate"]
)

color_diagnostics.sort_values(
    "return_rate_gap",
    ascending=False
)[
    [
        "articleID",
        "colorCode",
        "ordered_units",
        "returned_units",
        "return_rate",
        "article_return_rate",
        "return_rate_gap"
    ]
].head(20)

,articleID,colorCode,ordered_units,returned_units,return_rate,article_return_rate,return_rate_gap
19,i1001078,1081,628,426,0.678344,0.599407,0.078937
21,i1001082,1033,995,681,0.684422,0.612000,0.072422
5,i1000783,1007,488,311,0.637295,0.566067,0.071228
33,i1001674,1089,454,308,0.678414,0.607240,0.071174
60,i1001918,1081,436,280,0.642202,0.571429,0.070773
78,i1003886,1067,595,388,0.652101,0.584575,0.067526
73,i1003519,7103,232,149,0.642241,0.576909,0.065333
70,i1003519,7092,681,429,0.629956,0.576909,0.053047
49,i1001877,1033,823,594,0.721750,0.670940,0.050810
16,i1000868,1014,599,371,0.619366,0.573958,0.045408


In [58]:
variant_gap_summary = pd.DataFrame({
    "size": {
        "variants": len(size_diagnostics_robust),
        "mean_abs_gap": size_diagnostics_robust["return_rate_gap"].abs().mean(),
        "median_abs_gap": size_diagnostics_robust["return_rate_gap"].abs().median(),
        "gap_above_5pp": (size_diagnostics_robust["return_rate_gap"] > 0.05).mean(),
        "gap_above_10pp": (size_diagnostics_robust["return_rate_gap"] > 0.10).mean()
    },
    "color": {
        "variants": len(color_diagnostics),
        "mean_abs_gap": color_diagnostics["return_rate_gap"].abs().mean(),
        "median_abs_gap": color_diagnostics["return_rate_gap"].abs().median(),
        "gap_above_5pp": (color_diagnostics["return_rate_gap"] > 0.05).mean(),
        "gap_above_10pp": (color_diagnostics["return_rate_gap"] > 0.10).mean()
    }
}).T

variant_gap_summary

,variants,mean_abs_gap,median_abs_gap,gap_above_5pp,gap_above_10pp
size,127.0,0.025821,0.017871,0.070866,0.007874
color,85.0,0.030021,0.023458,0.105882,0.000000


In [59]:
size_article_spread = (
    size_diagnostics_robust
    .groupby("articleID")
    .agg(
        variants=("sizeCode", "nunique"),
        min_return_rate=("return_rate", "min"),
        max_return_rate=("return_rate", "max")
    )
)

size_article_spread["return_rate_spread"] = (
    size_article_spread["max_return_rate"]
    - size_article_spread["min_return_rate"]
)

size_article_spread.sort_values(
    "return_rate_spread",
    ascending=False
)

,variants,min_return_rate,max_return_rate,return_rate_spread
articleID,,,,
i1003890,7,0.570499,0.728395,0.157896
i1001078,6,0.537143,0.694737,0.157594
i1000868,6,0.500000,0.654545,0.154545
i1003886,7,0.525000,0.670330,0.145330
i1001774,6,0.664399,0.807692,0.143293
i1002334,6,0.730337,0.851093,0.120756
i1003519,10,0.508197,0.627907,0.119710
i1003505,10,0.547170,0.662722,0.115552
i1001813,6,0.628866,0.742690,0.113824


In [60]:
color_article_spread = (
    color_diagnostics
    .groupby("articleID")
    .agg(
        variants=("colorCode", "nunique"),
        min_return_rate=("return_rate", "min"),
        max_return_rate=("return_rate", "max")
    )
)

color_article_spread["return_rate_spread"] = (
    color_article_spread["max_return_rate"]
    - color_article_spread["min_return_rate"]
)

color_article_spread.sort_values(
    "return_rate_spread",
    ascending=False
)

,variants,min_return_rate,max_return_rate,return_rate_spread
articleID,,,,
i1001082,4,0.514778,0.684422,0.169644
i1000783,6,0.485095,0.637295,0.152200
i1001078,4,0.541995,0.678344,0.136349
i1003519,7,0.511521,0.642241,0.130721
i1003886,5,0.528757,0.652101,0.123344
i1001918,6,0.521311,0.642202,0.120890
i1001674,6,0.558904,0.678414,0.119510
i1001877,4,0.641584,0.721750,0.080166
i1000794,5,0.562025,0.641414,0.079389


In [61]:
(size_article_spread["return_rate_spread"] >= 0.10).sum(), \
(color_article_spread["return_rate_spread"] >= 0.10).sum()

(np.int64(10), np.int64(7))

In [62]:
top100_ids = set(top100_analysis["articleID"])

top100_transactions = product_base.loc[
    product_base["articleID"].isin(top100_ids)
].copy()

top100_transactions = top100_transactions.merge(
    top100_analysis[
        ["articleID", "stays_in_top100"]
    ],
    on="articleID",
    how="left"
)

In [63]:
markdown_comparison = (
    top100_transactions
    .groupby("stays_in_top100")
    .agg(
        lines=("articleID", "size"),
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        markdown_lines=(
            "price_status",
            lambda x: (x == "below_rrp").sum()
        ),
        avg_markdown_pct=("markdown_pct", "mean")
    )
)

In [64]:
markdown_only = (
    top100_transactions.loc[
        top100_transactions["price_status"] == "below_rrp"
    ]
    .groupby("stays_in_top100")
    .agg(
        markdown_lines=("articleID", "size"),
        avg_markdown_pct=("markdown_pct", "mean"),
        median_markdown_pct=("markdown_pct", "median"),
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum")
    )
)

markdown_only["return_rate"] = (
    markdown_only["returned_units"]
    / markdown_only["ordered_units"]
)

markdown_only

,markdown_lines,avg_markdown_pct,median_markdown_pct,ordered_units,returned_units,return_rate
stays_in_top100,,,,,,
False,19920,0.466893,0.499861,20039,11704,0.584061
True,56525,0.469818,0.499861,56995,25197,0.442091


In [65]:
markdown_comparison["markdown_share"] = (
    markdown_comparison["markdown_lines"]
    / markdown_comparison["lines"]
)

markdown_comparison["weighted_return_rate"] = (
    markdown_comparison["returned_units"]
    / markdown_comparison["ordered_units"]
)

markdown_comparison

,lines,ordered_units,returned_units,markdown_lines,avg_markdown_pct,markdown_share,weighted_return_rate
stays_in_top100,,,,,,,
False,50714,51066,31777,19920,0.183391,0.392791,0.622273
True,368643,378060,158059,56525,0.072038,0.153333,0.418079


In [66]:
price_status_comparison = (
    top100_transactions.loc[
        top100_transactions["price_status"].isin(
            ["full_price", "below_rrp"]
        )
    ]
    .groupby(
        ["stays_in_top100", "price_status"]
    )
    .agg(
        lines=("articleID", "size"),
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        kept_units=("kept_units", "sum")
    )
    .reset_index()
)

price_status_comparison["return_rate"] = (
    price_status_comparison["returned_units"]
    / price_status_comparison["ordered_units"]
)

price_status_comparison

,stays_in_top100,price_status,lines,ordered_units,returned_units,kept_units,return_rate
0,False,below_rrp,19920,20039,11704,8335,0.584061
1,False,full_price,30794,31027,20073,10954,0.646953
2,True,below_rrp,56525,56995,25197,31798,0.442091
3,True,full_price,308434,317364,130597,186767,0.411505


In [67]:
price_status_comparison["unit_share"] = (
    price_status_comparison["ordered_units"]
    / price_status_comparison.groupby(
        "stays_in_top100"
    )["ordered_units"].transform("sum")
)

price_status_comparison

,stays_in_top100,price_status,lines,ordered_units,returned_units,kept_units,return_rate,unit_share
0,False,below_rrp,19920,20039,11704,8335,0.584061,0.392414
1,False,full_price,30794,31027,20073,10954,0.646953,0.607586
2,True,below_rrp,56525,56995,25197,31798,0.442091,0.152247
3,True,full_price,308434,317364,130597,186767,0.411505,0.847753


In [68]:
price_by_product = (
    top100_transactions.loc[
        top100_transactions["price_status"].isin(
            ["full_price", "below_rrp"]
        )
    ]
    .groupby(
        ["articleID", "stays_in_top100", "price_status"]
    )
    .agg(
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum")
    )
    .reset_index()
)

price_by_product["return_rate"] = (
    price_by_product["returned_units"]
    / price_by_product["ordered_units"]
)

In [69]:
price_pivot = (
    price_by_product
    .pivot_table(
        index=["articleID", "stays_in_top100"],
        columns="price_status",
        values=["ordered_units", "return_rate"]
    )
)

price_pivot.columns = [
    "_".join(col).strip()
    for col in price_pivot.columns
]

price_pivot = price_pivot.reset_index()

In [70]:
price_pivot.columns

Index(['articleID', 'stays_in_top100', 'ordered_units_below_rrp',
       'ordered_units_full_price', 'return_rate_below_rrp',
       'return_rate_full_price'],
      dtype='str')

In [71]:
price_pivot_robust = price_pivot.loc[
    (price_pivot["ordered_units_full_price"] >= 50)
    &
    (price_pivot["ordered_units_below_rrp"] >= 50)
].copy()

In [72]:
price_pivot_robust["markdown_return_gap"] = (
    price_pivot_robust["return_rate_below_rrp"]
    - price_pivot_robust["return_rate_full_price"]
)

price_pivot_robust[
    [
        "articleID",
        "stays_in_top100",
        "ordered_units_full_price",
        "ordered_units_below_rrp",
        "return_rate_full_price",
        "return_rate_below_rrp",
        "markdown_return_gap"
    ]
].sort_values(
    "markdown_return_gap",
    ascending=False
)

,articleID,stays_in_top100,ordered_units_full_price,ordered_units_below_rrp,return_rate_full_price,return_rate_below_rrp,markdown_return_gap
70,i1001809,True,3318.0,337.0,0.649186,0.691395,0.042208
21,i1000420,True,4084.0,241.0,0.354554,0.385892,0.031338
97,i1003886,False,2557.0,127.0,0.583105,0.614173,0.031068
72,i1001853,True,2753.0,484.0,0.585906,0.609504,0.023598
22,i1000436,True,5788.0,159.0,0.302004,0.308176,0.006172
...,...,...,...,...,...,...,...
75,i1001906,False,1071.0,1463.0,0.647059,0.484621,-0.162438
60,i1001401,True,2469.0,593.0,0.551235,0.382799,-0.168436
93,i1003519,False,2308.0,377.0,0.601820,0.424403,-0.177417
10,i1000350,True,4099.0,144.0,0.531105,0.326389,-0.204716


In [73]:
price_pivot_robust.groupby(
    "stays_in_top100"
)["markdown_return_gap"].describe()

,count,mean,std,min,25%,50%,75%,max
stays_in_top100,,,,,,,,
False,19.0,-0.087934,0.054476,-0.177417,-0.114564,-0.100181,-0.058305,0.031068
True,62.0,-0.076131,0.052118,-0.211189,-0.110627,-0.077402,-0.039503,0.042208


## Conclusiones de Product Intelligence

El análisis confirma que incorporar las devoluciones cambia de forma relevante la lectura del rendimiento de producto.

### 1. Demanda bruta vs demanda retenida

Los rankings por `ordered_units` y `kept_units` están altamente correlacionados (`Spearman ≈ 0,95`), pero existen cambios relevantes en determinados productos.

La mediana del desplazamiento absoluto es de **150 posiciones**, con caídas de hasta **1.490 posiciones**.

### 2. Falsos bestsellers

De los 100 productos con mayor demanda bruta:

- **81 permanecen** en el Top 100 por unidades retenidas.
- **19 salen** del Top 100.

Estos 19 productos presentan una tasa de devolución ponderada de aproximadamente **62,2%**, frente a **41,8%** en los productos que permanecen.

### 3. Impacto en valor

La misma diferencia aparece en valor:

- productos que salen del Top 100: **63,3%** de devolución por valor;
- productos que permanecen: **46,7%**.

Esto demuestra que analizar únicamente ventas o demanda bruta puede sobrevalorar determinados productos.

### 4. Producto y variantes

Algunos `productGroup` están sobrerrepresentados entre los productos que pierden posiciones.

Talla y color también muestran diferencias dentro de un mismo artículo, aunque funcionan mejor como **drill-down de diagnóstico** que como explicación principal del problema.

### 5. Markdown

Los productos que salen del Top 100 tienen mayor exposición a markdown:

- ~**39%** de sus líneas están por debajo del RRP;
- frente a ~**15%** en los productos que permanecen.

Sin embargo, dentro del mismo artículo, el markdown no se asocia con una mayor tasa de devolución.

Por tanto, no se interpreta como causa directa de las devoluciones.

### Resultado

La combinación de demanda bruta, devoluciones y demanda retenida permite detectar productos que aparentan ser bestsellers pero cuyo rendimiento real se deteriora una vez incorporadas las devoluciones.

El siguiente paso será trasladar estas métricas a una capa visual orientada a decisiones de Merchandising.